In [ ]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# 載入資料
df = pd.read_csv("data/train.csv")

# 特徵工程
df['Sex'] = LabelEncoder().fit_transform(df['Sex'])
df['Age'].fillna(df['Age'].mean(), inplace=True)
df['Fare'].fillna(df['Fare'].mean(), inplace=True)

X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

# 切分資料
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立模型
model = LogisticRegression()
model.fit(X_train, y_train)

print("模型準確率:", model.score(X_test, y_test))

# 儲存模型

with open("model/titanic_model.pkl", "wb") as f:
    pickle.dump(model, f)


In [ ]:
import os
from openai import OpenAI
#from functions import query_data, plot_data, predict_survival
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    api_key=os.getenv(
        "OPENAI_API_KEY"
))


def ai_agent(user_input):
    # 判斷功能
    if "畫" in user_input or "圖" in user_input:
        img = plot_data(user_input)
        return {"type": "image", "content": img}
    elif "預測" in user_input or "會不會生還" in user_input:
        # 假設使用者會輸入特徵
        # ex: "25 歲女性 頭等艙 票價 80"
        import re

        age = int(re.search(r"(\d+)歲", user_input).group(1))
        sex = "女性" if "女性" in user_input else "男性"
        pclass = 1 if "頭等艙" in user_input else (2 if "二等艙" in user_input else 3)
        fare = float(re.search(r"票價\s*(\d+)", user_input).group(1))
        result = predict_survival(pclass, sex, age, fare)
        return {"type": "text", "content": result}
    else:
        answer = query_data(user_input)
        return {"type": "text", "content": answer}


In [ ]:
import streamlit as st
from agent import ai_agent
import base64

st.title("Titanic AI Agent 🚢")
st.write("使用者可以詢問 Titanic 資料、視覺化圖表，或進行生還預測。")

user_input = st.text_input(
    "請輸入問題，例如：'生還率多少？' 或 '畫出生還率和艙等關係' 或 '我是一位25歲女性，頭等艙，票價80，會不會生還？'"
)

if st.button("送出"):
    result = ai_agent(user_input)
    if result["type"] == "text":
        st.write(result["content"])
    elif result["type"] == "image":
        st.image(base64.b64decode(result["content"]), use_column_width=True)


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from functions import query_data, plot_data, predict_survival

load_dotenv()
client = OpenAI(
    api_key=os.getenv(
        "OPENAI_API_KEY"
)

# 定義 function 給 GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "query_data",
            "description": "回答關於 Titanic 資料的問題（例如生還率、性別差異）",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "問題，例如：生還率多少"}
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "plot_data",
            "description": "產生 Titanic 資料視覺化圖表",
            "parameters": {
                "type": "object",
                "properties": {
                    "plot_type": {"type": "string", "description": "圖表類型，例如：艙等、性別"}
                },
                "required": ["plot_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "predict_survival",
            "description": "預測乘客是否能生還",
            "parameters": {
                "type": "object",
                "properties": {
                    "pclass": {"type": "integer", "description": "艙等（1, 2, 3）"},
                    "sex": {"type": "string", "description": "性別：male 或 female"},
                    "age": {"type": "number", "description": "年齡"},
                    "fare": {"type": "number", "description": "票價"}
                },
                "required": ["pclass", "sex", "age", "fare"]
            }
        }
    }
]

def ai_agent(user_input):
    # 第一次請求：讓 GPT-4 判斷要呼叫哪個 function
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": user_input}],
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = eval(tool_call.function.arguments)

            if function_name == "query_data":
                result = query_data(arguments["question"])
            elif function_name == "plot_data":
                result = plot_data(arguments["plot_type"])
            elif function_name == "predict_survival":
                result = predict_survival(arguments["pclass"], arguments["sex"], arguments["age"], arguments["fare"])
            else:
                result = "未知功能"

        # 第二次請求：將結果交給 GPT-4 生成自然語言回應
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "你是一個 Titanic AI 助理"},
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": str(result)}
            ]
        )
        return {"type": "text", "content": final_response.choices[0].message.content}

    else:
        # GPT 沒有選擇 function，直接回應
        return {"type": "text", "content": message.content}
